# 01 — Tiền xử lý ảnh (miễn phí · OpenCV)

Chuẩn hoá ảnh: **xám hoá, tăng tương phản (CLAHE), khử nhiễu, làm thẳng (deskew)**, tuỳ chọn **nhị phân hoá**.

> **Với PaddleOCR** (bước 02), thường KHÔNG cần nhị phân mạnh — để `MODE='light'` (ảnh xám) là đủ,
> thậm chí có thể **bỏ qua notebook này** và OCR thẳng ảnh gốc. Chỉ dùng `MODE='binarize'` khi ảnh mờ/nhiễu nặng.

Vào: `data/raw_images/downloaded_images/<docid>/*.jpg` → Ra: `data/preprocessed/<docid>/*.png`.

In [ ]:
# %pip install opencv-python numpy matplotlib tqdm

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / 'src'))   # để import module trong src/

DATA_DIR    = PROJECT_ROOT / 'data'
RAW_IMG_DIR = DATA_DIR / 'raw_images' / 'downloaded_images'   # ảnh gốc theo <docid>/
PREP_DIR    = DATA_DIR / 'preprocessed'
OCR_DIR     = DATA_DIR / 'ocr'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
for d in [PREP_DIR, OCR_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def list_works(base):
    """Danh sách 'work id' = tên thư mục con trong base (mỗi tác phẩm/quyển = 1 folder)."""
    if not base.exists():
        return []
    return sorted(p.name for p in base.iterdir() if p.is_dir())

def images_of(base, wid):
    d = base / wid
    return sorted(list(d.glob('*.jpg')) + list(d.glob('*.png')))

print('Project root:', PROJECT_ROOT)
print('Ảnh gốc     :', RAW_IMG_DIR)

In [ ]:
import cv2, numpy as np
from tqdm.auto import tqdm

MODE = 'light'   # 'light' (khuyến nghị cho PaddleOCR) hoặc 'binarize'

def deskew(gray):
    inv = cv2.bitwise_not(gray)
    thr = cv2.threshold(inv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thr > 0))
    if len(coords) < 50:
        return gray
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.3:
        return gray
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def preprocess(img_bgr, mode='light'):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    gray = deskew(gray)
    if mode == 'binarize':
        gray = cv2.fastNlMeansDenoising(gray, h=10)
        return cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)
    return gray

In [ ]:
works = list_works(RAW_IMG_DIR)
print(f'{len(works)} tác phẩm/quyển:', works[:5], '...')

for wid in works:
    dst = PREP_DIR / wid; dst.mkdir(parents=True, exist_ok=True)
    for ip in tqdm(images_of(RAW_IMG_DIR, wid), desc=wid, leave=False):
        out = dst / (ip.stem + '.png')
        if out.exists():
            continue
        img = cv2.imread(str(ip))
        if img is None:
            print('  ! đọc lỗi:', ip); continue
        cv2.imwrite(str(out), preprocess(img, MODE))
print('Xong. Ảnh ở:', PREP_DIR)

## So sánh trước / sau (1 ảnh mẫu)

In [ ]:
import matplotlib.pyplot as plt
works = list_works(RAW_IMG_DIR)
if works:
    s = images_of(RAW_IMG_DIR, works[0])[0]
    before = cv2.cvtColor(cv2.imread(str(s)), cv2.COLOR_BGR2RGB)
    after = preprocess(cv2.imread(str(s)), MODE)
    fig, ax = plt.subplots(1, 2, figsize=(12, 8))
    ax[0].imshow(before); ax[0].set_title('Gốc'); ax[0].axis('off')
    ax[1].imshow(after, cmap='gray'); ax[1].set_title(f'Sau ({MODE})'); ax[1].axis('off')
    plt.show()